# Gemma 3N Video Captioning (Gradio)
Upload a video and generate one caption from your finetuned model.

This app shows:
- Input prompt used for inference
- Generated caption
- Time taken (seconds)

In [ ]:
!pip install -U transformers
!pip install accelerate opencv-python gradio -q


In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # Optional: add token if model is private
if HF_TOKEN.strip():
    login(token=HF_TOKEN)

In [ ]:
%%capture
!pip install --no-deps --upgrade timm  # Only for Gemma 3N
!pip uninstall torchcodec -y


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

processor = AutoProcessor.from_pretrained("blind-assist/google-gemma-3n-2b-e3")
model = AutoModelForImageTextToText.from_pretrained(
    "blind-assist/google-gemma-3n-2b-e3",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded on: {model.device}")


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import time
from pathlib import Path
from urllib.parse import unquote

def _normalize_candidate_path(candidate):
    if candidate is None:
        return None

    if isinstance(candidate, Path):
        candidate = str(candidate)

    if isinstance(candidate, os.PathLike):
        candidate = os.fspath(candidate)

    if not isinstance(candidate, str):
        return None

    text = candidate.strip()
    if not text:
        return None

    if text.startswith("file="):
        text = text.split("file=", 1)[1]
    elif "file=" in text:
        text = text.split("file=", 1)[1]

    text = unquote(text)

    if text.startswith("file://"):
        text = text.replace("file://", "", 1)

    return text

def resolve_video_path(video_value):
    if video_value is None:
        return None

    normalized = _normalize_candidate_path(video_value)
    if normalized:
        return normalized

    if isinstance(video_value, (list, tuple)):
        for item in video_value:
            resolved = resolve_video_path(item)
            if resolved:
                return resolved
        return None

    if isinstance(video_value, dict):
        preferred_keys = ["path", "video", "name", "value", "file", "url"]
        for key in preferred_keys:
            if key in video_value:
                resolved = resolve_video_path(video_value[key])
                if resolved:
                    return resolved

        for _, value in video_value.items():
            resolved = resolve_video_path(value)
            if resolved:
                return resolved
        return None

    for attr in ["path", "name", "video", "value", "file", "url"]:
        if hasattr(video_value, attr):
            try:
                resolved = resolve_video_path(getattr(video_value, attr))
                if resolved:
                    return resolved
            except Exception:
                pass

    if hasattr(video_value, "__dict__"):
        for _, value in vars(video_value).items():
            resolved = resolve_video_path(value)
            if resolved:
                return resolved

    return None

def cache_video_path(video_value):
    return resolve_video_path(video_value)

def downsample_video(video_path, num_frames=2):
    """Extracts evenly spaced frames for VLM context."""
    if not video_path or not os.path.exists(video_path):
        return []

    vidcap = cv2.VideoCapture(video_path)
    if not vidcap.isOpened():
        vidcap.release()
        return []

    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        vidcap.release()
        return []

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, image = vidcap.read()
        if success:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(image))
    vidcap.release()
    return frames

instruction = (
    "Given the visual input from the user's forward perspective, "
    "generate exactly one short sentence to guide a visually impaired user "
    "by identifying critical obstacles or landmarks, describing their locations "
    "using clock directions relative to the user (12 o'clock is straight ahead), "
    "including relevant details such as size, material, or distance, and giving "
    "one clear action, while prioritizing immediate safety and avoiding any extra explanation."
)

def caption_video(video_value, cached_video_path=None):
    video_path = resolve_video_path(video_value) or cached_video_path
    if not video_path:
        return instruction, f"No video file was received (type: {type(video_value).__name__}). Please upload or record again.", "0.00"

    frames = downsample_video(video_path)
    if not frames:
        return instruction, "Could not extract frames from this video. Try uploading an MP4 or recording again.", "0.00"

    content = []
    for img in frames:
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": instruction})

    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)

    start_time = time.time()

    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

    inference_time = round(time.time() - start_time, 3)

    generated_text = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    guidance = " ".join(generated_text.strip().split())

    return instruction, guidance, f"{inference_time:.2f}"


In [ ]:
import gradio as gr

with gr.Blocks(title="Gemma 3N Video Captioning") as demo:
    gr.Markdown("## 🎥 Gemma 3N Video Captioning")

    video_path_state = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            video_in = gr.Video(
                label="Upload Video",
                sources=["upload", "webcam"],
                format="mp4",
            )
            run_btn = gr.Button("Generate Caption", variant="primary")

        with gr.Column(scale=1):
            prompt_out = gr.Textbox(label="Input Prompt", lines=4, interactive=False)
            caption_out = gr.Textbox(label="Generated Caption", lines=4, interactive=False)
            time_out = gr.Textbox(label="Time Taken (seconds)", interactive=False)

    video_in.change(
        fn=cache_video_path,
        inputs=[video_in],
        outputs=[video_path_state],
    )

    run_btn.click(
        fn=caption_video,
        inputs=[video_in, video_path_state],
        outputs=[prompt_out, caption_out, time_out],
    )

demo.launch(share=True, debug=True)
